# Session 3: 위치 데이터 피처 엔지니어링

**집값 예측 대회** - DevOps 엔지니어를 위한 ML 교육 커리큘럼 (5회 중 3회차)

---

이번 세션에서는 위도(lat)와 경도(long) 데이터를 활용하여 새로운 피처를 생성하는 방법을 배웁니다.

집값을 결정하는 가장 중요한 요소는 **입지(위치)**입니다. 같은 면적의 집이라도 어디에 있느냐에 따라 가격이 크게 달라지죠. 이번 세션에서 배울 세 가지 기법을 통해 위치 정보를 더 효과적으로 모델에 전달할 수 있습니다.

## 목차

### 이론 (20분)
1. PCA (주성분 분석) - 좌표 변환
2. K-Means 클러스터링 - 동네 만들기
3. Haversine 거리 - 이웃 찾기

### 실습 (70분)
- Step 1: 데이터 로딩 + 헬퍼 함수 정의
- Step 2: PCA 변환
- Step 3: K-Means 클러스터링
- Step 4: Haversine 거리

### 연습 문제

---

# 이론 (20분)

## 1. PCA (Principal Component Analysis, 주성분 분석)

### 핵심 개념

PCA는 원래 **차원 축소**에 주로 사용되는 알고리즘이지만, 여기서는 다른 용도로 활용합니다.

위도(lat)와 경도(long)는 2차원 데이터입니다. 이 2차원 데이터를 **차원 축소 없이 2차원 그대로 PCA 변환**하면, 원본 좌표를 회전/스케일링한 새로운 좌표계를 얻을 수 있습니다.

### 왜 유용한가?

- 원본 위도/경도와는 다른 축으로 데이터를 바라보게 됩니다
- 데이터의 분산이 가장 큰 방향으로 좌표가 재정렬됩니다
- 트리 모델(LightGBM)은 축에 평행한 분할만 가능한데, PCA로 축을 회전시키면 **대각선 방향의 패턴**도 포착할 수 있습니다

### 직관적 이해

```
원본 좌표 (lat, long)          PCA 변환 후 (pca1, pca2)
┌─────────────────┐           ┌─────────────────┐
│    *  *          │           │         *  *     │
│  *  *  *         │  ──PCA──> │       *  *  *    │
│ *  *  * *        │           │     *  *  * *    │
│*  *  *   *       │           │   *  *  *        │
└─────────────────┘           └─────────────────┘
같은 데이터, 다른 시각          분산이 큰 방향으로 정렬
```

## 2. K-Means 클러스터링

### 핵심 개념

K-Means는 **비지도 학습** 알고리즘으로, 데이터를 K개의 그룹(클러스터)으로 나눕니다.

위도/경도 데이터에 K-Means를 적용하면 **가까운 집들끼리 같은 그룹**으로 묶이기 때문에, 우편번호(zipcode)와 유사하지만 데이터 기반으로 자동 생성되는 "동네" 피처를 만들 수 있습니다.

### K 값 결정: 두 가지 방법

K-Means에서 가장 중요한 결정은 **K (클러스터 수)를 몇으로 할 것인가**입니다.

#### 방법 1: Elbow Method

K를 늘려가면서 **inertia**(각 데이터 포인트와 소속 클러스터 중심 간 거리의 합)를 측정합니다. inertia가 급격히 떨어지다가 완만해지는 "팔꿈치" 지점의 K를 선택합니다.

```
Inertia
│\                    
│ \                   
│  \                  
│   \.____  <-- 여기가 Elbow!
│        \______
│               \____
└──────────────────── K
```

#### 방법 2: CV Score 기반 선택

K-Means로 만든 클러스터 피처를 실제 모델에 넣고 **CV Score가 가장 좋은 K**를 선택합니다.

Elbow Method는 클러스터링 자체의 품질을 보지만, CV Score 방법은 **최종 목표(집값 예측)에 대한 성능**으로 판단하기 때문에 실전에서 더 좋은 결과를 주는 경우가 많습니다.

| 방법 | 기준 | 장점 | 단점 |
|------|------|------|------|
| Elbow Method | 클러스터 내 거리 | 빠르고 직관적 | 최적 K가 모호할 수 있음 |
| CV Score | 최종 모델 성능 | 목표에 직접 최적화 | 시간이 오래 걸림 |

## 3. Haversine 거리

### 핵심 개념

Haversine 공식은 **지구의 곡률을 고려**하여 두 좌표(위도, 경도) 간의 실제 거리(km)를 계산합니다.

단순히 유클리드 거리를 쓰면 안 되는 이유는, 지구가 구형이기 때문입니다. 위도 1도의 거리와 경도 1도의 거리는 위치에 따라 다릅니다.

### DevOps 비유: CDN 엣지 서버 선택

Haversine 거리는 **CDN(Content Delivery Network)에서 사용자에게 가장 가까운 엣지 서버를 찾는 것**과 동일한 원리입니다.

```
CDN 시나리오:                    집값 예측 시나리오:
┌─────────────────┐             ┌─────────────────┐
│  사용자 요청      │             │  특정 집 (id=0)   │
│     │            │             │     │            │
│     v            │             │     v            │
│ 가장 가까운       │             │ 가장 가까운       │
│ 엣지 서버 선택    │             │ 이웃집 탐색       │
│     │            │             │     │            │
│     v            │             │     v            │
│ 빠른 응답 제공    │             │ 이웃 통계 피처 생성│
└─────────────────┘             └─────────────────┘
```

CDN이 사용자와 가까운 서버에서 콘텐츠를 제공하는 것처럼, 특정 집과 가까운 이웃집들의 정보(평균 가격, 평균 면적 등)를 피처로 활용하면 예측 성능이 크게 향상됩니다.

### 수학 공식

$$d = 2r \cdot \arcsin\left(\sqrt{\sin^2\left(\frac{\phi_2 - \phi_1}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\lambda_2 - \lambda_1}{2}\right)}\right)$$

여기서 $r$은 지구 반지름(6371km), $\phi$는 위도, $\lambda$는 경도입니다.

---

# 실습 (70분)

## Step 1: 데이터 로딩 + 헬퍼 함수 정의

먼저 필요한 라이브러리를 불러오고, 이번 세션에서 반복적으로 사용할 헬퍼 함수들을 정의합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import lightgbm as lgb

# 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

# 시각화 설정
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 8)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

### 헬퍼 함수 정의

아래 함수들은 이번 세션 전체에서 반복 사용됩니다.

- `rmse_exp()`: log 변환된 예측값을 원래 스케일로 되돌린 후 RMSE를 계산합니다
- `load_original_data()`: 학습/테스트 데이터를 로딩하고, 이상치 제거 및 log 변환을 수행합니다
- `train_test_split_custom()`: 학습/테스트 분리 및 One-Hot Encoding을 처리합니다
- `get_oof_lgb()`: LightGBM 5-Fold CV를 수행하고 OOF 예측값과 점수를 반환합니다

In [ ]:
def rmse_exp(y_true, y_pred):
    """log1p 변환된 값을 expm1으로 원래 스케일로 되돌린 후 RMSE 계산"""
    return np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(y_pred)))


def load_original_data():
    """원본 데이터 로딩, 이상치 제거, log 변환 수행"""
    train = pd.read_csv('../input/train.csv')
    test = pd.read_csv('../input/test.csv')

    train_copy = train.copy()
    train_copy['data'] = 'train'
    test_copy = test.copy()
    test_copy['data'] = 'test'
    test_copy['price'] = np.nan

    # 이상치 제거: 면적이 매우 넓은데 가격이 낮은 데이터
    train_copy = train_copy[
        ~((train_copy['sqft_living'] > 12000) & (train_copy['price'] < 3000000))
    ].reset_index(drop=True)

    # 학습/테스트 데이터 합치기
    data = pd.concat([train_copy, test_copy], sort=False).reset_index(drop=True)
    data = data[train_copy.columns]

    # 날짜 컬럼 제거, zipcode를 문자열로 변환
    data.drop('date', axis=1, inplace=True)
    data['zipcode'] = data['zipcode'].astype(str)

    # 가격 로그 변환 (치우침 보정)
    skew_columns = ['price']
    for c in skew_columns:
        data[c] = np.log1p(data[c])

    return data


def train_test_split_custom(data, do_ohe=True):
    """학습/테스트 분리 및 원핫 인코딩 처리"""
    df = data.drop(['id', 'price', 'data'], axis=1).copy()
    cat_cols = df.select_dtypes('object').columns

    for col in cat_cols:
        if do_ohe:
            # 원핫 인코딩
            ohe_df = pd.get_dummies(df[[col]], prefix='ohe_' + col)
            df.drop(col, axis=1, inplace=True)
            df = pd.concat([df, ohe_df], axis=1)
        else:
            # 레이블 인코딩
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])

    # 학습/테스트 분리
    train_len = data[data['data'] == 'train'].shape[0]
    X_train = df.iloc[:train_len]
    X_test = df.iloc[train_len:]
    y_train = data[data['data'] == 'train']['price']

    return X_train, X_test, y_train


def get_oof_lgb(X_train, y_train, X_test, lgb_param,
                verbose_eval=False, return_cv_score_only=False):
    """LightGBM 5-Fold CV OOF 예측
    
    return_cv_score_only=True이면 CV 점수만 반환합니다.
    False이면 (oof, predictions, cv_score, feature_importance_df)를 반환합니다.
    """
    folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    oof = np.zeros(len(X_train))
    predictions = np.zeros(len(X_test))
    feature_importance_df = pd.DataFrame()

    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train.values, y_train.values)):
        if verbose_eval > 0:
            print(f'Fold : {fold_ + 1}')

        trn_data = lgb.Dataset(X_train.iloc[trn_idx], label=y_train.iloc[trn_idx])
        val_data = lgb.Dataset(X_train.iloc[val_idx], label=y_train.iloc[val_idx])

        num_round = 100000
        clf = lgb.train(
            lgb_param, trn_data, num_round,
            valid_sets=[trn_data, val_data],
            verbose_eval=verbose_eval,
            early_stopping_rounds=200
        )

        oof[val_idx] = clf.predict(X_train.iloc[val_idx], num_iteration=clf.best_iteration)
        predictions += clf.predict(X_test, num_iteration=clf.best_iteration) / folds.n_splits

        cv_fold_score = rmse_exp(y_train.iloc[val_idx], oof[val_idx])
        if verbose_eval > 0:
            print(f'Fold {fold_ + 1} / CV-Score: {cv_fold_score:.6f}')

        # 피처 중요도 기록
        fold_importance_df = pd.DataFrame()
        fold_importance_df['feature'] = X_train.columns.tolist()
        fold_importance_df['importance'] = clf.feature_importance('gain')
        fold_importance_df['fold'] = fold_ + 1
        feature_importance_df = pd.concat([feature_importance_df, fold_importance_df], axis=0)

    cv_score = rmse_exp(y_train, oof)
    print(f'CV-Score: {cv_score:.6f}')

    if return_cv_score_only:
        return cv_score
    else:
        return oof, predictions, cv_score, feature_importance_df

### 피처 중요도 시각화 함수

In [ ]:
def plot_feature_importance(fi_df, num_feature=20):
    """피처 중요도 상위 N개를 막대 그래프로 시각화"""
    cols = (
        fi_df[['feature', 'importance']]
        .groupby('feature')
        .mean()
        .sort_values(by='importance', ascending=False)[:num_feature]
        .index
    )
    best_features = fi_df.loc[fi_df.feature.isin(cols)]

    sns.barplot(
        x='importance', y='feature',
        data=best_features.sort_values(by='importance', ascending=False)
    )
    plt.title('Feature Importances (5-fold 평균)')
    plt.tight_layout()
    plt.show()

### 데이터 로딩 및 확인

In [ ]:
data = load_original_data()

print('데이터 크기:', data.shape)
data.head()

### Baseline 모델

새로운 피처를 추가할 때마다 성능이 개선되는지 확인하기 위해, 먼저 기본 데이터로 **Baseline CV Score**를 측정합니다.

LightGBM 5-Fold CV를 사용합니다.

In [ ]:
X_train, X_test, y_train = train_test_split_custom(data)
print('학습 데이터:', X_train.shape, '/ 테스트 데이터:', X_test.shape)

# LightGBM 하이퍼파라미터
lgb_param = {
    'objective': 'regression',
    'learning_rate': 0.05,
    'num_leaves': 15,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'feature_fraction': 0.7,
    'seed': RANDOM_SEED,
    'metric': ['rmse'],
}

# Baseline CV Score 측정
oof, pred, cv_score, fi_df = get_oof_lgb(X_train, y_train, X_test, lgb_param)
print(f'\nBaseline CV-Score: {cv_score:,.0f}')

---

## Step 2: PCA 변환

위도(lat)와 경도(long) 데이터를 PCA로 변환하여 `coord_pca1`, `coord_pca2`라는 새로운 피처를 생성합니다.

2차원 -> 2차원 변환이므로 차원 축소가 아니라, **좌표계의 회전**에 가깝습니다.

In [ ]:
# 원본 데이터 다시 로딩
data = load_original_data()

# 위도, 경도 추출
coord = data[['lat', 'long']]

# PCA 변환: 2차원 -> 2차원 (축 회전)
pca = PCA(n_components=2)
coord_pca = pca.fit_transform(coord)

# 새로운 피처로 추가
data['coord_pca1'] = coord_pca[:, 0]
data['coord_pca2'] = coord_pca[:, 1]

print('PCA 변환 완료!')
print(f'원본: lat={data["lat"].iloc[0]:.4f}, long={data["long"].iloc[0]:.3f}')
print(f'PCA:  pca1={data["coord_pca1"].iloc[0]:.4f}, pca2={data["coord_pca2"].iloc[0]:.4f}')

### PCA 변환 결과 시각화

PCA로 변환된 좌표를 시각화하면, 원본 지도와 형태는 비슷하지만 축이 회전된 것을 확인할 수 있습니다. 색상은 집값(price)을 나타냅니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 원본 좌표
sns.scatterplot(x='long', y='lat', hue='price', data=data, ax=axes[0], alpha=0.5)
axes[0].set_title('원본 좌표 (lat, long)')
axes[0].legend([], frameon=False)

# PCA 변환 좌표
sns.scatterplot(x='coord_pca2', y='coord_pca1', hue='price', data=data, ax=axes[1], alpha=0.5)
axes[1].set_title('PCA 변환 좌표 (pca1, pca2)')
axes[1].legend([], frameon=False)

plt.tight_layout()
plt.show()

### PCA 피처 추가 후 CV Score 확인

PCA 피처를 추가했을 때 모델 성능이 향상되는지 확인합니다.

In [ ]:
X_train, X_test, y_train = train_test_split_custom(data)
print('학습 데이터:', X_train.shape, '/ 테스트 데이터:', X_test.shape)

lgb_param = {
    'objective': 'regression',
    'learning_rate': 0.05,
    'num_leaves': 15,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'feature_fraction': 0.7,
    'seed': RANDOM_SEED,
    'metric': ['rmse'],
}

oof, pred, cv_score, fi_df = get_oof_lgb(X_train, y_train, X_test, lgb_param)
print(f'\nPCA 추가 후 CV-Score: {cv_score:,.0f}')
print('Baseline 대비 개선되었는지 확인하세요!')

PCA 피처를 추가한 것만으로도 CV Score가 개선됩니다. 단 두 개의 피처를 추가했을 뿐인데 효과가 있다는 것은, 위도/경도 데이터에 축 회전 변환이 유효한 정보를 제공한다는 의미입니다.

---

## Step 3: K-Means 클러스터링

위도/경도 데이터에 K-Means 클러스터링을 적용하여 데이터 기반의 "동네" 피처를 생성합니다.

핵심은 **최적의 K 값을 찾는 것**입니다. 두 가지 방법을 비교해보겠습니다.

### 3-1. Elbow Method로 K 결정

K를 2부터 15까지 늘려가면서 inertia 변화를 관찰합니다.

In [ ]:
# 위도, 경도 좌표 추출
coord = data[['lat', 'long']]

# K를 2~15까지 변화시키며 inertia 측정
inertia_arr = []
k_range = range(2, 16)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED).fit(coord)
    interia = kmeans.inertia_  # 클러스터 내 거리 합
    print(f'K={k:2d}  inertia: {interia:.2f}')
    inertia_arr.append(interia)

inertia_arr = np.array(inertia_arr)

In [ ]:
# Elbow Method 시각화
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_arr, 'o-')
plt.vlines(5, ymin=inertia_arr.min() * 0.9999, ymax=inertia_arr.max() * 1.0003,
           linestyles='--', colors='b', label='K=5 (Elbow)')
plt.title('Elbow Method - 최적 K 탐색')
plt.xlabel('클러스터 수 (K)')
plt.ylabel('Inertia (클러스터 내 거리 합)')
plt.legend()
plt.show()

그래프를 보면 K=5 부근에서 inertia 감소 속도가 완만해지는 것을 확인할 수 있습니다. Elbow Method에 따르면 **K=5**가 적절합니다.

하지만 이것이 정말 최적일까요? CV Score 기반으로도 확인해보겠습니다.

### 3-2. CV Score 기반 K 결정

K-Means 클러스터 피처를 실제 모델에 넣어 CV Score가 가장 좋은 K를 찾습니다.

먼저 넓은 범위(K=2, 7, 12, ..., 77)에서 대략적인 최적 영역을 탐색합니다.

In [ ]:
# 1단계: 넓은 범위에서 탐색 (K=2, 7, 12, ..., 77)
k_range_wide = range(2, 80, 5)
scores_wide = []

for k in k_range_wide:
    # K-Means 클러스터링 수행
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED).fit(coord)
    coord_cluster = kmeans.predict(coord)
    data['coord_cluster'] = coord_cluster
    data['coord_cluster'] = data['coord_cluster'].map(lambda x: str(x).rjust(2, '0'))

    # 학습/테스트 분리 후 CV Score 측정
    X_train, X_test, y_train = train_test_split_custom(data)

    lgb_param = {
        'objective': 'regression',
        'learning_rate': 0.05,
        'num_leaves': 15,
        'bagging_fraction': 0.7,
        'bagging_freq': 1,
        'feature_fraction': 0.7,
        'seed': RANDOM_SEED,
        'metric': ['rmse'],
    }

    print(f'K = {k}')
    score = get_oof_lgb(X_train, y_train, X_test, lgb_param, return_cv_score_only=True)
    scores_wide.append((k, score))
    print()

넓은 범위 탐색 결과, K=32 부근이 가장 좋은 성능을 보입니다. 이제 K=28~36 범위에서 **세밀하게 탐색**합니다.

In [ ]:
# 2단계: 좁은 범위에서 세밀하게 탐색 (K=28..36)
k_range_narrow = range(28, 37)
scores_narrow = []

for k in k_range_narrow:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED).fit(coord)
    coord_cluster = kmeans.predict(coord)
    data['coord_cluster'] = coord_cluster
    data['coord_cluster'] = data['coord_cluster'].map(lambda x: str(x).rjust(2, '0'))

    X_train, X_test, y_train = train_test_split_custom(data)

    lgb_param = {
        'objective': 'regression',
        'learning_rate': 0.05,
        'num_leaves': 15,
        'bagging_fraction': 0.7,
        'bagging_freq': 1,
        'feature_fraction': 0.7,
        'seed': RANDOM_SEED,
        'metric': ['rmse'],
    }

    print(f'K = {k}')
    score = get_oof_lgb(X_train, y_train, X_test, lgb_param, return_cv_score_only=True)
    scores_narrow.append((k, score))
    print()

### 결론: K=32가 최적

세밀 탐색 결과 **K=32**일 때 CV Score가 가장 좋습니다.

| 방법 | 최적 K | 비고 |
|------|--------|------|
| Elbow Method | K=5 | 클러스터 품질 기준 |
| CV Score | K=32 | 최종 모델 성능 기준 |

최종 목표가 집값 예측이므로, **CV Score 기준의 K=32**를 선택합니다.

### K=32 클러스터 시각화

K=32로 클러스터링한 결과를 지도 위에 시각화합니다.

In [ ]:
# 최종 K=32로 클러스터링 적용
data = load_original_data()
coord = data[['lat', 'long']]

kmeans = KMeans(n_clusters=32, random_state=RANDOM_SEED).fit(coord)
coord_cluster = kmeans.predict(coord)
data['coord_cluster'] = coord_cluster
data['coord_cluster'] = data['coord_cluster'].map(lambda x: 'c_' + str(x).rjust(2, '0'))

# 클러스터 지도 시각화
plt.figure(figsize=(14, 10))
sns.scatterplot(
    x='long', y='lat', hue='coord_cluster',
    hue_order=np.sort(data['coord_cluster'].unique()),
    data=data, alpha=0.6
)
plt.title('K-Means 클러스터링 (K=32) - 데이터 기반 동네 구분')
plt.xlabel('경도 (Longitude)')
plt.ylabel('위도 (Latitude)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

### 클러스터 피처 추가 후 Feature Importance 확인

K=32 클러스터 피처를 추가했을 때의 성능과 피처 중요도를 확인합니다.

In [ ]:
X_train, X_test, y_train = train_test_split_custom(data)
print('학습 데이터:', X_train.shape, '/ 테스트 데이터:', X_test.shape)

lgb_param = {
    'objective': 'regression',
    'learning_rate': 0.05,
    'num_leaves': 15,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'feature_fraction': 0.7,
    'seed': RANDOM_SEED,
    'metric': ['rmse'],
}

oof, pred, cv_score, fi_df = get_oof_lgb(X_train, y_train, X_test, lgb_param)
print(f'\nK-Means(K=32) 추가 후 CV-Score: {cv_score:,.0f}')

In [ ]:
# 피처 중요도 시각화
plot_feature_importance(fi_df)

피처 중요도를 보면 특정 클러스터(예: c_11)가 상위에 올라오는 것을 확인할 수 있습니다. 이는 해당 클러스터(동네)가 집값 예측에 중요한 정보를 가지고 있다는 의미입니다.

---

## Step 4: Haversine 거리

마지막으로 **Haversine 공식**을 사용하여 두 좌표 간의 실제 거리(km)를 계산합니다.

DevOps 관점에서 생각하면, CDN이 사용자 위치에서 가장 가까운 엣지 서버를 찾듯이, 각 집에서 가장 가까운 이웃집들을 찾아 그 정보를 피처로 활용하는 것입니다.

In [ ]:
def haversine_array(lat1, lng1, lat2, lng2):
    """Haversine 공식으로 두 좌표 간 거리(km)를 계산
    
    지구의 곡률을 고려한 거리 계산입니다.
    numpy 배열을 지원하므로 벡터 연산이 가능합니다.
    """
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    AVG_EARTH_RADIUS = 6371  # km
    lat = lat2 - lat1
    lng = lng2 - lng1
    d = np.sin(lat * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(lng * 0.5) ** 2
    h = 2 * AVG_EARTH_RADIUS * np.arcsin(np.sqrt(d))
    return h

### 데이터 내 최대 거리 확인

데이터에 포함된 집들 사이의 최대 거리가 얼마나 되는지 확인합니다.

In [ ]:
# 위도/경도의 최소, 최대 좌표 간 거리 = 데이터 내 최대 거리
print(f'위도 범위: {data["lat"].min():.4f} ~ {data["lat"].max():.4f}')
print(f'경도 범위: {data["long"].min():.3f} ~ {data["long"].max():.3f}')

max_dist = haversine_array(
    data['lat'].min(), data['long'].min(),
    data['lat'].max(), data['long'].max()
)
print(f'\n데이터 내 최대 거리: {max_dist:.2f}km')

약 **113.88km** 범위 내의 데이터입니다.

### 특정 집(id=0)에서 모든 집까지의 거리 계산

id=0인 집을 기준으로, 전체 집까지의 거리를 계산해봅니다. 이것이 "이웃 탐색"의 기본 원리입니다.

In [ ]:
# id=0 집의 좌표
lat1 = data.loc[0, 'lat']   # id=0 집의 위도
long1 = data.loc[0, 'long'] # id=0 집의 경도

# 전체 집의 좌표
lat2 = data['lat'].values
long2 = data['long'].values

# Haversine 거리 계산 (벡터 연산으로 한 번에 처리)
dist_arr = haversine_array(lat1, long1, lat2, long2)

# 결과를 DataFrame으로 정리
neighbor_df = pd.DataFrame({
    'id': np.tile(np.array([data.loc[0, 'id']]), data.shape[0]),
    'neighbor_id': data['id'],
    'neighbor_lat': lat2,
    'neighbor_long': long2,
    'distance': dist_arr,
})

print(f'id=0 집의 좌표: ({lat1}, {long1})')
print(f'이웃 데이터 크기: {neighbor_df.shape}')
print()
neighbor_df.head()

### 반경 5km 이내 이웃 시각화

id=0 집을 기준으로 반경 5km 이내의 이웃들을 지도 위에 시각화합니다.

CDN 비유로 말하면, 이것은 특정 서버의 **5km 서비스 반경** 내에 있는 사용자를 찾는 것과 같습니다.

In [ ]:
# 5km 이내 이웃 여부 표시
neighbor_df['within_5km'] = neighbor_df['distance'] <= 5

print(f'전체 집 수: {len(neighbor_df)}')
print(f'5km 이내 이웃 수: {neighbor_df["within_5km"].sum()}')
print(f'비율: {neighbor_df["within_5km"].mean() * 100:.1f}%')

# 시각화
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='neighbor_long', y='neighbor_lat',
    hue='within_5km', data=neighbor_df, alpha=0.5
)

# id=0 집 위치 강조
plt.scatter([long1], [lat1], c='red', s=200, marker='*', zorder=5, label='id=0 집')

plt.title('id=0 집 기준 반경 5km 이내 이웃')
plt.xlabel('경도 (Longitude)')
plt.ylabel('위도 (Latitude)')
plt.legend()
plt.show()

### 이웃 통계 피처 (Neighbor Stat Features) 개념

Haversine 거리의 진짜 가치는 여기서 나옵니다. 각 집에 대해 가까운 이웃들을 찾은 후, 그 이웃들의 **통계 정보**를 피처로 만들 수 있습니다.

예를 들어, 특정 집의 반경 5km 이내 이웃들에 대해:

| 피처 | 설명 | 직관 |
|------|------|------|
| `neighbor_mean_price` | 이웃 평균 가격 | 비싼 동네인가? |
| `neighbor_median_sqft` | 이웃 면적 중앙값 | 큰 집이 많은 지역인가? |
| `neighbor_count` | 이웃 수 | 밀집 지역인가? |
| `neighbor_mean_grade` | 이웃 평균 등급 | 고급 주택 밀집 지역인가? |

**DevOps 비유**: CDN에서 특정 리전의 트래픽 패턴(평균 요청 수, 피크 시간 등)을 분석하는 것과 유사합니다. 가까운 서버들의 부하 통계를 기반으로 라우팅을 최적화하듯, 가까운 집들의 통계를 기반으로 가격을 예측합니다.

이웃 통계 피처 생성의 상세한 구현은 다음 노트북들을 참조하세요:
- `notebook/Generate Neighbor Info.ipynb` - 이웃 정보 생성
- `notebook/Generate Neighbor Stat.ipynb` - 이웃 통계 생성

이 노트북들에서는 전체 데이터에 대해 Haversine 거리를 계산하고, 다양한 반경 기준으로 이웃 통계 피처를 생성하는 전체 파이프라인을 구현합니다.

---

## 전체 요약

이번 세션에서 배운 세 가지 위치 기반 피처 엔지니어링 기법:

| 기법 | 입력 | 출력 | 효과 |
|------|------|------|------|
| **PCA 변환** | lat, long | coord_pca1, coord_pca2 | 좌표 축 회전으로 새로운 패턴 포착 |
| **K-Means 클러스터링** | lat, long | coord_cluster | 데이터 기반 "동네" 피처 생성 |
| **Haversine 거리** | lat, long | distance (km) | 이웃 탐색 및 통계 피처 기반 |

---

## 연습 문제

아래 연습 문제를 풀어보면서 이번 세션의 내용을 복습하세요.

### 연습 문제 1: PCA 변환 심화

`lat`, `long`에 `sqft_living`을 추가하여 3차원 데이터를 PCA로 2차원으로 축소해보세요. 기존 2차원 PCA보다 CV Score가 개선되는지 확인해보세요.

**힌트**: `PCA(n_components=2)`를 사용하되, 입력 데이터에 `sqft_living` 컬럼을 추가합니다. `StandardScaler`로 스케일링을 먼저 하면 더 좋은 결과를 얻을 수 있습니다.

In [ ]:
# 연습 문제 1: 여기에 코드를 작성하세요
# from sklearn.preprocessing import StandardScaler
# ...


### 연습 문제 2: 다른 반경의 이웃 탐색

id=0 집에 대해 **반경 1km, 3km, 10km** 이내의 이웃 수를 각각 세어보세요. 반경에 따라 이웃 수가 어떻게 변하는지 막대 그래프로 시각화해보세요.

**힌트**: Step 4에서 계산한 `neighbor_df`의 `distance` 컬럼을 사용하세요.

In [ ]:
# 연습 문제 2: 여기에 코드를 작성하세요
# radii = [1, 3, 5, 10]
# ...


### 연습 문제 3: K-Means + PCA 결합

PCA 변환 피처(`coord_pca1`, `coord_pca2`)와 K-Means 클러스터 피처(`coord_cluster`, K=32)를 **동시에** 추가했을 때의 CV Score를 측정해보세요. 각각 따로 추가했을 때보다 성능이 더 좋아지나요?

**힌트**: `load_original_data()`로 깨끗한 데이터를 로딩한 후, PCA와 K-Means 피처를 모두 추가합니다.

In [ ]:
# 연습 문제 3: 여기에 코드를 작성하세요
# data = load_original_data()
# coord = data[['lat', 'long']]
# ...


### 연습 문제 4: DBSCAN 클러스터링 비교

K-Means 대신 **DBSCAN** 클러스터링을 사용하여 동네 피처를 생성해보세요. DBSCAN은 클러스터 수(K)를 미리 지정하지 않아도 되는 장점이 있습니다. K-Means(K=32)와 CV Score를 비교해보세요.

**힌트**: `from sklearn.cluster import DBSCAN`을 사용하세요. `eps`(이웃 반경)와 `min_samples`(최소 포인트 수) 파라미터를 조정해야 합니다.

In [ ]:
# 연습 문제 4: 여기에 코드를 작성하세요
# from sklearn.cluster import DBSCAN
# ...
